## Thiết lập môi trường và kiến thức cơ bản về hệ sinh thái

In [8]:
!pip install accelerate bitsandbytes google-cloud-aiplatform peft trl scikit-learn tokenizers torch transformers unsloth evaluate python-dotenv sentencepiece protobuf



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [9]:
!nvidia-smi

Wed May  6 18:20:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 576.88                 Driver Version: 576.88         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060      WDDM  |   00000000:84:00.0  On |                  N/A |
|  0%   47C    P8             16W /  170W |     871MiB /  12288MiB |     36%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
import os
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from huggingface_hub import login
from dotenv import load_dotenv

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'transformers'

In [ ]:
print(torch.cuda.is_available())

### Khởi tạo và đăng nhập

In [ ]:
load_dotenv()
hf_token = os.getenv("HUGFACE_TOKEN")
if hf_token:
    login(token=hf_token)
else: 
    print("Warning: HUGFACE_TOKEN not found in .env")


### Xác định bài toán và lấy dữ liẹu

In [ ]:
print("Loading dataset: tridm/UIT-VSMEC...")
dataset = load_dataset("tridm/UIT-VSMEC")
dataset

In [ ]:
labels = ['Other', 'Disgust', 'Enjoyment', 'Anger', 'Surprise', 'Sadness', 'Fear']
label2id = {label: i for i,label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

In [ ]:
def preprocess_function(examples):
    examples["label"] = [label2id[l] for l in examples["Emotion"]]
    return examples

In [ ]:
print("Mapping labels to IDs...")
dataset = dataset.map(preprocess_function, batched=True)

### Tokenization và tiền xử lý

In [ ]:
model_name = "FPTAI/vibert-base-cased"
print(f"Loading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenizer_function(examples):
    return tokenizer(examples["Sentence"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing dataset...")
tokenized_datasets = dataset.map(tokenizer_function, batched=True)

### Chọn mô hình pre-trained

In [ ]:
print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

### Các chỉ số đánh giá

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

### Huấn luyện Arguments và Trainer

In [ ]:
repo_name = "vibert-vsmec-emotion-recognition"

training_args = TrainingArguments(
    output_dir=repo_name,
    eval_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    push_to_hub=True,
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

### Fine-tuning

In [ ]:
print("Training...")
trainer.train()

### Đẩy lên Hugging Face Hub

In [ ]:
print("Push model to Hugging Face Hub...")
trainer.push_to_hub()